# B1: Grid Alignment

Worked example of aligning input rasters onto the common 0.1° target grid.
- Input: GEE exports, CROPGRIDS, TRENDY and other rasters at differing resolutions
- Method: bilinear interpolation / nearest-neighbour resampling
- Output: nothing is written to disk. This notebook demonstrates the alignment and the
  annual→monthly PSN-fraction disaggregation for one year; the production stages
  (B3–B5, C1) apply the same `reproject_match` call inline on their own inputs.

In [ ]:
import yaml
from pathlib import Path
import sys
sys.path.append(str(Path('../src')))

import numpy as np
import xarray as xr
import rioxarray
from grid_utils import create_target_grid
from rasterio.enums import Resampling

with open(Path('../config.yaml')) as f:
    cfg = yaml.safe_load(f)

In [ ]:
da_template = create_target_grid()

In [ ]:
for k, v in cfg['paths'].items():
    cfg["paths"][k] = Path(v)#.expanduser().resolve()
    # pass

In [ ]:
da_npp_annual = (
    xr.open_dataarray(cfg['paths']['gee_exports'] / "NPP_annual_2010_2025.tif")
        .rio.write_crs("EPSG:4326")\
        .rio.reproject_match(da_template, resampling=Resampling.nearest)
        .drop_attrs()
        .rename({'band': 'year'})
        .rename("npp_annual")
        .assign_coords(year=np.arange(2010, 2026))
)

In [ ]:
year = 2015
da_PSN_monthly = (
    xr.open_dataarray(cfg['paths']['gee_exports'] / f"PSN_monthly_{year}.tif")
        .rio.write_crs("EPSG:4326")\
        .rio.reproject_match(da_template, resampling=Resampling.nearest)
        .drop_attrs()
        .rename({'band': 'month'})
        .rename("PSN_monthly")
        .assign_coords(month=np.arange(1, 13))
)

In [ ]:
da_PSN_monthly_ratio = da_PSN_monthly / da_PSN_monthly.sum(dim='month')
da_PSN_monthly_ratio = da_PSN_monthly_ratio.fillna(1/12)

In [ ]:
da_NPP_monthly = da_npp_annual.sel(year=year) * da_PSN_monthly_ratio